In [104]:
import json
from datetime import datetime, timedelta
import pandas as pd
import glob
import os
from sentence_transformers import SentenceTransformer, util
import torch
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import random
import re

## Setting Directory

In [3]:
os.chdir("../..")

## Importing Training Data with Manual Classes

In [46]:
filepath = f"data/mst/00a2_nyt_articles_sample_annotated.csv"

for_train = pd.read_csv(filepath)

In [50]:
# grabbing the indices of the training df
# because the training sample is unbalanced (too few 1s), we will subset the training sample to only include some of the 0s
for_train_pos = for_train[for_train['poli_con_class'] == 1]
random.seed(123)
for_train_neg = for_train[for_train['poli_con_class'] == 0].sample(n = 100)

for_train_subset = for_train_pos.append(for_train_neg)

In [51]:
for_train_subset_index = for_train_subset['news_index']

## Importing NYT embeddings

In [52]:
nyt_embeddings = np.load(f"data/mst/sentence_embeddings/01_nyt_embeddings.npy")

In [53]:
# making embeddings into a df n articles x n features (384)

nyt_embeddings_df = pd.DataFrame(nyt_embeddings)

nyt_train = nyt_embeddings_df.iloc[for_train_subset_index]

nyt_train

,0,1,2,3,4,5,6,7,8,9,...,374,375,376,377,378,379,380,381,382,383
3225,0.018272,0.121695,-0.000491,-0.061723,-0.009381,-0.025043,0.018593,-0.017052,0.032987,0.109989,...,0.045308,0.015760,-0.044934,-0.032976,-0.078296,0.066870,0.108081,-0.027799,0.039484,0.017752
1313,0.018298,0.111364,-0.097082,-0.021782,-0.002422,0.003657,0.009458,-0.025546,-0.041569,0.077598,...,0.071217,-0.000670,-0.037127,-0.004959,-0.014049,0.121537,0.044737,0.002519,0.018859,-0.073336
11808,0.116425,0.126671,0.063427,0.123462,0.069650,0.063068,-0.012853,0.063826,-0.037879,-0.076253,...,-0.004789,0.071493,0.010346,0.093218,0.001916,-0.052419,-0.009487,-0.092594,-0.062904,0.049626
3527,-0.081777,-0.019811,-0.078835,0.013259,0.004396,0.008245,0.033538,-0.094078,-0.048917,0.014519,...,0.050774,-0.085371,-0.022701,0.027132,0.046475,-0.027251,-0.068395,-0.019147,0.022959,0.021488
794,0.035394,-0.023382,-0.018196,0.053336,0.025043,0.071662,-0.066607,-0.036295,-0.040446,0.009018,...,-0.034441,0.068876,0.038917,0.041218,0.017483,0.066887,-0.007989,-0.083752,0.051832,0.062881
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6439,0.001341,-0.062448,-0.052917,0.002704,-0.080165,0.061193,0.088185,-0.040445,-0.052988,-0.055688,...,-0.005875,-0.000173,-0.070704,-0.020759,0.031254,0.029130,0.013342,0.051484,-0.060458,0.054488
8704,-0.028473,-0.102060,-0.043945,0.015397,-0.071617,0.046232,0.042939,0.027562,-0.043046,-0.013115,...,0.049046,-0.024531,-0.035494,0.055523,-0.014617,-0.017813,0.053881,-0.071090,0.079511,0.090507
8213,0.047950,-0.006188,0.088629,0.029262,-0.005927,0.034441,-0.167494,-0.051719,-0.089857,-0.025472,...,0.084166,0.020695,0.089619,-0.009747,-0.014693,0.065256,0.017407,0.000980,-0.076338,0.103646
10174,-0.018409,-0.044037,0.002102,0.036599,-0.015073,0.012269,-0.048145,-0.013528,0.021215,0.008979,...,0.087480,0.011759,-0.037538,-0.021411,0.033634,0.000631,-0.039981,-0.135140,-0.048093,0.038162


In [54]:
# Array of classifications for the training df n articles x 1 

In [56]:
nyt_class = for_train_subset['poli_con_class']

nyt_class

1      1
2      1
6      1
30     1
58     1
      ..
196    0
260    0
49     0
274    0
7      0
Name: poli_con_class, Length: 130, dtype: int64

In [57]:
## Training a simple logistic regression to classify texts relevant to political/conflict 

In [113]:
# importing sklearn library
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
# from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.model_selection import cross_validate
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report
from sklearn.metrics import precision_score, recall_score

In [72]:
# creating a train test split

# X_train, X_test, y_train, y_test = train_test_split(nyt_train, nyt_class, test_size=0.2, random_state=42)

# Set up 5-fold CV

cv = StratifiedKFold(n_splits=8, shuffle=True, random_state=42)

In [80]:
# logistic reg model
nyt_classification = LogisticRegression(max_iter=1000)

# svm model
nyt_classification_svm = SVC(kernel='linear', probability=True, class_weight='balanced') 


# fitting model
nyt_classification_svm.fit(X_train, y_train)


# getting performance metrics
scoring = ['accuracy', 'precision', 'recall', 'f1']

scores = cross_validate(nyt_classification_svm, nyt_train, nyt_class, cv=cv, scoring=scoring) 

In [95]:
# scores.get("test_accuracy").mean()
scores.get("test_precision").mean()

0.8541666666666666

In [ ]:
# looking at predicted vs actual 

In [96]:
y_pred = cross_val_predict(nyt_classification_svm, nyt_train, nyt_class, cv=5)
print(classification_report(y_pred, nyt_class))

              precision    recall  f1-score   support

           0       0.94      0.94      0.94       100
           1       0.80      0.80      0.80        30

    accuracy                           0.91       130
   macro avg       0.87      0.87      0.87       130
weighted avg       0.91      0.91      0.91       130



## Trying a simple lexicon-based approach instead

In [124]:
# defining eligible lexicon

poli_conflict_lexicon_short = np.array(["defense",  "ammunition", "armed", "army", "armistice", "battle",  "bomb", "artillery",  
                                  "destroy", "destruct", "conflict", "skirmsh" , 
                                 "grenade",  "guerrilla",    
                                 "military", "militant", "militia", "missile", "munition", "rebel", 
                                  "rocket",  "soldier", "troop", "war", "weapon","combat",  
                                  "humanitarian",  "invade", "invasion", 
                                 "junta", "territory", "retaliat", "attack", "casualt", "assault", 
                                   "drone", "escalat", "p o w", "explod", "explos",  "kill", "injure", "hostage", 
                                  "junta", "violen",  "fight", "ceasefire", "cease fire"])

# Join with | to create a regex OR pattern
poli_con_regex = "|".join(poli_conflict_lexicon_short)

In [125]:
# fn to clean string

def remove_special_chars_re(text):
    # This pattern keeps only alphanumeric characters and spaces
    # [^a-zA-Z0-9 ] means "any character NOT in the set of a-z, A-Z, 0-9, or space"
    cleaned_text = re.sub(r'[^a-zA-Z0-9 ]', ' ', text).lower()
    
    return cleaned_text

In [126]:
# creating a cleaned text col 
for_train_subset['cleaned_text'] = for_train_subset['nyt_title'] + "; " + for_train_subset['nyt_abstract']

for_train_subset['cleaned_text'] = for_train_subset['cleaned_text'].apply(lambda x: remove_special_chars_re(x))

In [127]:
for_train_subset['lexicon_based'] = for_train_subset['cleaned_text'].apply(lambda x: 1 if re.search(poli_con_regex, x) else 0)

In [128]:
print(for_train_subset[['lexicon_based', 'poli_con_class']])

     lexicon_based  poli_con_class
1                1               1
2                0               1
6                1               1
30               1               1
58               1               1
..             ...             ...
196              0               0
260              0               0
49               0               0
274              0               0
7                0               0

[130 rows x 2 columns]


In [129]:
precision_lexicon = precision_score(for_train_subset['poli_con_class'], for_train_subset['lexicon_based'])
recall_lexicon = recall_score(for_train_subset['poli_con_class'], for_train_subset['lexicon_based'])

print(f"Precision: {precision_lexicon:.2f}")
print(f"Recall: {recall_lexicon:.2f}")

Precision: 0.62
Recall: 0.93


In [131]:
pd.set_option('display.max_colwidth', None)

print(for_train_subset[(for_train_subset['lexicon_based'] == 1) & (for_train_subset['poli_con_class'] == 0)]['cleaned_text'])

234    takeaways from the hearing in the georgia trump case  fani t  willis  the district attorney  defended her personal conduct in a tense courtroom appearance as defense lawyers sought to disqualify her from the prosecution of donald j  trump and his allies in georgia 
211                                                      a cyberattack on a unitedhealth unit disrupts prescription drug orders  for a week  people have been waylaid at pharmacies after a unit of the nation s largest insurer was shut down by a possible ransomware assault 
12                                                                                 killer mike calls his grammys arrest a  speed bump   the artist was arrested on a misdemeanor battery charge after winning awards for best rap album  best rap performance and best rap song 
9                                                                                                      6 great space images in january  a rocket launching from the ocean  an asteroi